### Data Profiling

#### Understanding the HM Land registry data before EDA and modelling.

#### Importing Packages

In [51]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 50)

In [5]:
project_root = Path(".").resolve().parent
raw_path = project_root/"data"/"raw"/"pp-2025.csv"

#### HM Land Registry has no header rows so we are providing column names ourselves.

In [7]:
PP_COLUMNS = [
    "transaction_id", # Unique ID for transactions
    "price", # Sales Price in £ (Target Variable)
    "date_of_transfer", # Date of Completion
    "postcode", # Full Postcode
    "property_type", # D=Detached, S=Semi, T=Terraced, O=Other
    "old_new", # Y=New Building, N=Existing
    "duration", # F=Freehold, L=Leasehold, U=Unknown
    "paon", # Primary Address (Number, Building Name)
    "saon", # Secondary Address (Flat, Unit)
    "street", # Street name
    "locality", # District Name, Locality
    "town_city", # Town or City
    "district", # Local Authority District (example: London Borough)
    "county", # County Name
    "ppd_category", # A=Standard Sale, B=Non-Standard
    "record_status" # A=Addition, C=Change, D=Deletion
]

#### Loading and Exploring the Data

In [14]:
df = pd.read_csv(raw_path,
                 header=None,
                 names=PP_COLUMNS,
                 encoding="latin-1") # HM Land Registry uses Latin-1 encoding, not UTF-8

In [16]:
df.shape

(802761, 16)

In [17]:
df.head()

,transaction_id,price,date_of_transfer,postcode,property_type,old_new,duration,paon,saon,street,locality,town_city,district,county,ppd_category,record_status
0,{49E87C31-9D5F-591C-E063-4704A8C00C31},560000,04/12/2025 00:00,PE1 2QU,D,N,F,10,NaN,ALL SAINTS ROAD,NaN,PETERBOROUGH,CITY OF PETERBOROUGH,CITY OF PETERBOROUGH,A,A
1,{49E87C31-9D60-591C-E063-4704A8C00C31},230000,12/12/2025 00:00,PE2 9RY,T,N,F,19,NaN,OSWALD ROAD,NaN,PETERBOROUGH,CITY OF PETERBOROUGH,CITY OF PETERBOROUGH,A,A
2,{49E87C31-9D61-591C-E063-4704A8C00C31},272000,15/12/2025 00:00,PE16 6BL,D,N,F,2,NaN,SADDLERS WAY,NaN,CHATTERIS,FENLAND,CAMBRIDGESHIRE,A,A
3,{49E87C31-9D63-591C-E063-4704A8C00C31},249950,17/12/2025 00:00,CB4 3RY,T,N,L,72,NaN,COCKERELL ROAD,NaN,CAMBRIDGE,CAMBRIDGE,CAMBRIDGESHIRE,A,A
4,{49E87C31-9D65-591C-E063-4704A8C00C31},650000,12/12/2025 00:00,CB1 3BH,T,N,F,21,NaN,CAVENDISH PLACE,NaN,CAMBRIDGE,CAMBRIDGE,CAMBRIDGESHIRE,A,A


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 802761 entries, 0 to 802760
Data columns (total 16 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   transaction_id    802761 non-null  str  
 1   price             802761 non-null  int64
 2   date_of_transfer  802761 non-null  str  
 3   postcode          801314 non-null  str  
 4   property_type     802761 non-null  str  
 5   old_new           802761 non-null  str  
 6   duration          802761 non-null  str  
 7   paon              802761 non-null  str  
 8   saon              92055 non-null   str  
 9   street            790461 non-null  str  
 10  locality          306443 non-null  str  
 11  town_city         802761 non-null  str  
 12  district          802761 non-null  str  
 13  county            802761 non-null  str  
 14  ppd_category      802761 non-null  str  
 15  record_status     802761 non-null  str  
dtypes: int64(1), str(15)
memory usage: 98.0 MB


In [23]:
# Missing Values
missing_counts = df.isnull().sum()
missing_pct = round((df.isnull().sum()/len(df))*100, 2)
missing_counts, missing_pct

(transaction_id           0
 price                    0
 date_of_transfer         0
 postcode              1447
 property_type            0
 old_new                  0
 duration                 0
 paon                     0
 saon                710706
 street               12300
 locality            496318
 town_city                0
 district                 0
 county                   0
 ppd_category             0
 record_status            0
 dtype: int64,
 transaction_id       0.00
 price                0.00
 date_of_transfer     0.00
 postcode             0.18
 property_type        0.00
 old_new              0.00
 duration             0.00
 paon                 0.00
 saon                88.53
 street               1.53
 locality            61.83
 town_city            0.00
 district             0.00
 county               0.00
 ppd_category         0.00
 record_status        0.00
 dtype: float64)

In [29]:
df.describe()

,price
count,8.027610e+05
mean,3.756437e+05
std,1.198532e+06
min,1.000000e+00
25%,1.855600e+05
50%,2.840000e+05
75%,4.250000e+05
max,7.930200e+08


In [31]:
df.price.describe().apply(lambda x: f"{x:,.0f}")

count        802,761
mean         375,644
std        1,198,532
min                1
25%          185,560
50%          284,000
75%          425,000
max      793,020,000
Name: price, dtype: str

In [38]:
# Categorical Columns

categorical_cols = ["property_type", "old_new", "duration", "ppd_category", "record_status"]

for col in categorical_cols:
    counts = df[col].value_counts(dropna=False)
    pct = (df[col].value_counts(normalize=True, dropna=False) * 100).round(2)
    summary = pd.DataFrame({"count": counts, "pct": pct})
    print(f"\n--- {col} ---")
    print(summary)


--- property_type ---
                count    pct
property_type               
T              231367  28.82
S              225765  28.12
D              183890  22.91
F              131593  16.39
O               30146   3.76

--- old_new ---
          count    pct
old_new               
N        771735  96.14
Y         31026   3.86

--- duration ---
           count    pct
duration               
F         628808  78.33
L         173953  21.67

--- ppd_category ---
               count    pct
ppd_category               
A             686976  85.58
B             115785  14.42

--- record_status ---
                count    pct
record_status               
A              802761  100.0


In [44]:
# Geographical Distribution

top_counties = df["county"].value_counts().head(15)
top_districts = df["district"].value_counts().head(15)

print("Top 15 Counties:\n")
print (top_counties)

print ("\n\nTop 15 districts:\n")
print (top_districts)

Top 15 Counties:

county
GREATER LONDON        87589
GREATER MANCHESTER    37380
WEST YORKSHIRE        31616
WEST MIDLANDS         29776
KENT                  22969
ESSEX                 22426
HAMPSHIRE             20129
LANCASHIRE            19952
MERSEYSIDE            19156
SOUTH YORKSHIRE       17829
SURREY                16595
HERTFORDSHIRE         15421
TYNE AND WEAR         15106
NORFOLK               14214
NOTTINGHAMSHIRE       13211
Name: count, dtype: int64


Top 15 districts:

district
BIRMINGHAM                  10816
LEEDS                       10745
NORTH YORKSHIRE             10086
COUNTY DURHAM                8865
SOMERSET                     8747
CORNWALL                     8536
BUCKINGHAMSHIRE              7666
WILTSHIRE                    7451
CHESHIRE EAST                7215
BRADFORD                     7067
WEST NORTHAMPTONSHIRE        6528
SHEFFIELD                    6345
DORSET                       6308
CITY OF BRISTOL              6247
EAST RIDING OF YORKSHIR

In [48]:
# London
london_mask = df["county"] == "GREATER LONDON"
london_count = london_mask.sum()
london_pct = london_mask.mean()*100

print (f"Greater London sales: {london_count}")
print (f"Greater London Percentage: {london_pct}")

Greater London sales: 87589
Greater London Percentage: 10.91096851989571


#### Filtering:
Filter to London rows only: 87k rows out of 800k

Filter to Standard Sales: "ppd_category = A". Drop non-standard sales like reposessions for fair model market prices

Filter to Addition Records: "record_status = A". Drop changes and deletions

Filter Price Range: Keep price range between £50,000 and £10,000,000


#### Type Conversions:

Convert "date_of_transfer" to datetime format

Convert categorical columns to category dtype


#### Drop Columns

"transaction_id": Unique random ID, has no predictive value

"paon, saon, street, town_city, county": Address details too granular for our models. For town and county, after filtering all values are London

"locality": 61% missing and redundant with postcode and district

"ppd_category, record_status": After filtering all values are constant

Apply log-transformation to price for modeling:
Prices are right-skewed and log-transform stabilises variance and lets the model focus on relative error.